### Expense assistant
Here's the assistant that tells helps with your finances.

In [155]:
from openai import OpenAI
import gradio as gr
import sqlite3

In [156]:
openai = OpenAI()
ollama = OpenAI(api_key="", base_url="http://localhost:11434/v1")
system_prompt = "You are a snarky and mean assistant that manages a user's finances. You can add, view, update, and delete transactions, and provide summaries of the user's financial data."

In [157]:
bank = "money.db"
with sqlite3.connect(bank) as connection:
    cursor = connection.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS transactions (id INTEGER PRIMARY KEY, item TEXT, amount REAL, date TEXT)")
    connection.commit()
 

In [158]:
def add_transaction(item, amount, date):
    print("Database add transaction tool called with item:", item, "amount:", amount, "date:", date)
    with sqlite3.connect(bank) as connection:
        cursor = connection.cursor()
        cursor.execute("INSERT INTO transactions (item, amount, date) VALUES (?, ?, ?)", (item, amount, date))
        connection.commit()
    return "Transaction added successfully!"

def get_transactions():
    print("Database get transactions tool called")
    with sqlite3.connect(bank) as connection:
        cursor = connection.cursor()
        cursor.execute("SELECT * FROM transactions")
        transactions = cursor.fetchall()
    return transactions

def get_transaction_by_id(transaction_id):
    print("Database get transaction by ID tool called with ID:", transaction_id)
    with sqlite3.connect(bank) as connection:
        cursor = connection.cursor()
        cursor.execute("SELECT * FROM transactions WHERE id = ?", (transaction_id,))
        transaction = cursor.fetchone()
    return transaction

def update_transaction(transaction_id, item=None, amount=None, date=None):
    updates = []
    values = []

    if item is not None:
        updates.append("item = ?")
        values.append(item)

    if amount is not None:
        updates.append("amount = ?")
        values.append(amount)

    if date is not None:
        updates.append("date = ?")
        values.append(date)

    if not updates:
        return "Nothing to update."

    values.append(transaction_id)

    query = f"""
        UPDATE transactions
        SET {", ".join(updates)}
        WHERE id = ?
    """

    cursor.execute(query, values)
    connection.commit()
    return "Transaction updated successfully!"

def delete_transaction(transaction_id):
    print("Database delete transaction tool called with ID:", transaction_id)
    with sqlite3.connect(bank) as connection:
        cursor = connection.cursor()
        cursor.execute("DELETE FROM transactions WHERE id = ?", (transaction_id,))
        connection.commit()
    return "Transaction deleted successfully!"

def delete_everything():
    print("Database delete everything tool called")
    with sqlite3.connect(bank) as connection:
        cursor = connection.cursor()
        cursor.execute("DELETE FROM transactions")
        connection.commit()
    return "All transactions deleted successfully!"

In [159]:
add_transaction_function = {
    "type": "function",
    "function": {
        "name": "add_transaction",
        "description": "Add a new transaction to the database.",
        "parameters": {
            "type": "object",
            "properties": {
                "item": {
                    "type": "string",
                    "description": "The item for the transaction",
                },
                "amount": {
                    "type": "number",
                    "description": "The amount for the transaction",
                },
                "date": {
                    "type": "string",
                    "description": "The date of the transaction",
                }
            },
            "required": ["item", "amount", "date"],
            "additionalProperties": False
        }
    }
}


get_transactions_function = {
    "type": "function",
    "function": {
        "name": "get_transactions",
        "description": "Retrieve all transactions from the database.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        }
    }
}


get_transaction_by_id_function = {
    "type": "function",
    "function": {
        "name": "get_transaction_by_id",
        "description": "Retrieve a transaction by its ID from the database.",
        "parameters": {
            "type": "object",
            "properties": {
                "transaction_id": {
                    "type": "integer",
                    "description": "The ID of the transaction to retrieve",
                }
            },
            "required": ["transaction_id"],
            "additionalProperties": False
        }
    }
}


update_transaction_function = {
    "type": "function",
    "function": {
        "name": "update_transaction",
        "description": "Update one or more fields of an existing transaction in the database.",
        "parameters": {
            "type": "object",
            "properties": {
                "transaction_id": {
                    "type": "integer",
                    "description": "The ID of the transaction to update",
                },
                "item": {
                    "type": "string",
                    "description": "The new item for the transaction",
                },
                "amount": {
                    "type": "number",
                    "description": "The new amount for the transaction",
                },
                "date": {
                    "type": "string",
                    "description": "The new date of the transaction",
                }
            },
            "required": ["transaction_id"],
            "additionalProperties": False
        }
    }
}


delete_transaction_function = {
    "type": "function",
    "function": {
        "name": "delete_transaction",
        "description": "Delete a transaction from the database.",
        "parameters": {
            "type": "object",
            "properties": {
                "transaction_id": {
                    "type": "integer",
                    "description": "The ID of the transaction to delete",
                }
            },
            "required": ["transaction_id"],
            "additionalProperties": False
        }
    }
}

tools = [
    add_transaction_function,
    get_transactions_function,
    get_transaction_by_id_function,
    update_transaction_function,
    delete_transaction_function
]

In [ ]:
import json

def chat(message, history):

    history = [{"role": h["role"], "content": h["content"]} for h in history]

    system_message = {"role": "system", "content": system_prompt}

    messages = [system_message] + history + [{"role": "user", "content": message}]

    print("Chat function called with message:", message)

    response = ollama.chat.completions.create(model="llama3.2",messages=messages,tools=tools)

    print("TOOL CALL:", response.choices[0].message.tool_calls)

    while response.choices[0].message.tool_calls:

        messages.append(response.choices[0].message)

        for tool_call in response.choices[0].message.tool_calls:
            print("Tool call detected:", tool_call.function.name, "with arguments:", tool_call.function.arguments)
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            
            print("*" * 20)
            print(response.choices[0].message)
            print("*" * 20)


            tool_result = handle_tool_call(tool_name, tool_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(tool_result)
            })
        
        print("*" * 20)
        print(messages)
        print("*" * 20)
        

        response = ollama.chat.completions.create(
            model="llama3.2",
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content


In [161]:
def handle_tool_call(tool_name, tool_args):
    if tool_name == "add_transaction":
        return add_transaction(**tool_args)
    elif tool_name == "get_transactions":
        return get_transactions()
    elif tool_name == "get_transaction_by_id":
        return get_transaction_by_id(**tool_args)
    elif tool_name == "update_transaction":
        return update_transaction(**tool_args)
    elif tool_name == "delete_transaction":
        return delete_transaction(**tool_args)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

In [162]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


In [163]:
print(get_transactions())

Database get transactions tool called
[]


Chat function called with message: Hi
TOOL CALL: None
Chat function called with message: If you want my respect, you have to speak politely to me
TOOL CALL: None
Chat function called with message: I bought a cell phone today, and a wristwatch yesterday
TOOL CALL: None
Chat function called with message: the phone cost 300 euros and 100 for the wristwatch
TOOL CALL: [ChatCompletionMessageFunctionToolCall(id='call_HKL4Xy0x9Yk0nKQ7FIa2G7ku', function=Function(arguments='{"item": "Cell phone", "amount": 300, "date": "2023-10-06"}', name='add_transaction'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_1U1lX7QCPiWJzjH27y3VjLiS', function=Function(arguments='{"item": "Wristwatch", "amount": 100, "date": "2023-10-05"}', name='add_transaction'), type='function')]
Tool call detected: add_transaction with arguments: {"item": "Cell phone", "amount": 300, "date": "2023-10-06"}
********************
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[],